# 07 — Updated Ranking: Input Corrections + Calibration

This notebook produces the final updated global river plastic emission ranking.

## Approach

Instead of re-running Meijer's grid-level model (which failed at catchment level, R²=-0.03),
we apply **ratio corrections** to Meijer's published emissions where we have direct evidence
that input data was wrong, then layer the **observational calibration** on top.

### Correction layers
1. **Plastic fraction**: Meijer assumed 12% constant; WaW 3.0 has country-specific values (1.6–30%)
2. **Observational calibration**: log₁₀(obs) = 0.72 × log₁₀(Meijer) + 0.62 (slope < 1, n=64)

### Why ratio approach, not full re-run?
Meijer runs at 30-arcsec grid cells with multiplicative probabilities P(M)×P(R)×P(O).
Computing these at catchment level with averaged inputs violates Jensen's inequality
(average of nonlinear function ≠ nonlinear function of average), producing R²=-0.03.
The ratio approach preserves Meijer's grid-level spatial structure and only corrects
inputs where we have direct evidence they're wrong.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

DATA_PROC = Path("../data/processed")
DATA_RAW = Path("../data/raw")
RESULTS = Path("../results/figures")
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. Load data

In [ ]:
fm = pd.read_csv(DATA_PROC / "feature_matrix_v1.csv")
cal = pd.read_csv(DATA_PROC / "recalibrated_emissions_v1.csv")
obs = pd.read_csv(DATA_PROC / "observed_flux_matched_S3.csv")

df = fm.copy()
df['meijer_ton_yr'] = cal['meijer_ton_yr']
df['calibrated_ton_yr'] = cal['calibrated_ton_yr']
df['rank_meijer'] = cal['rank_meijer']
df['rank_calibrated'] = cal['rank_calibrated']

print(f'Rivers: {len(df):,}')
print(f'Meijer total: {df["meijer_ton_yr"].sum():,.0f} ton/yr')
print(f'Calibrated total: {df["calibrated_ton_yr"].sum():,.0f} ton/yr')
print(f'Observed rivers: {len(obs)}')

## 2. Plastic fraction correction

Meijer assumed plastic = 12% of municipal solid waste for ALL countries.
WaW 3.0 reports country-specific plastic fractions ranging from 1.6% to 30%.

In [ ]:
# plastic_pct in feature matrix is in percentage form (e.g. 10.55 for Philippines)
# Meijer assumed 12% → ratio = (plastic_pct / 100) / 0.12 = plastic_pct / 12
df['plastic_ratio'] = df['plastic_pct'] / 12.0

print('Plastic fraction correction factor (WaW 3.0 / Meijer 12%):')
print(f'  Mean: {df["plastic_ratio"].mean():.3f}')
print(f'  Range: [{df["plastic_ratio"].min():.3f}, {df["plastic_ratio"].max():.3f}]')
print(f'  No change (=1.0): {(abs(df["plastic_ratio"] - 1.0) < 0.01).sum():,} rivers ({(abs(df["plastic_ratio"] - 1.0) < 0.01).mean()*100:.1f}%)')
print()

# Apply correction
df['E_plastic_corrected'] = df['meijer_ton_yr'] * df['plastic_ratio']
df['rank_plastic'] = df['E_plastic_corrected'].rank(ascending=False)

print(f'Plastic-corrected total: {df["E_plastic_corrected"].sum():,.0f} ton/yr')
print(f'Change vs Meijer: {(df["E_plastic_corrected"].sum() / df["meijer_ton_yr"].sum() - 1)*100:+.1f}%')

In [ ]:
# Country-level impact
country_impact = df.groupby('country_iso').agg(
    meijer=('meijer_ton_yr', 'sum'),
    corrected=('E_plastic_corrected', 'sum'),
    plastic_ratio=('plastic_ratio', 'first')
).sort_values('meijer', ascending=False)

print('TOP 15 COUNTRIES — plastic_pct correction impact:')
print(f'{"Country":7s} {"Meijer":>10s} {"Corrected":>10s} {"Ratio":>6s} {"Change":>10s}')
print('-'*48)
for iso, row in country_impact.head(15).iterrows():
    if pd.isna(iso):
        continue
    ch = row['corrected'] - row['meijer']
    print(f'{iso:7s} {row["meijer"]:>10,.0f} {row["corrected"]:>10,.0f} {row["plastic_ratio"]:>6.2f} {ch:>+10,.0f}')

In [ ]:
# Top 20 rivers — ranking changes
top20_meijer = df.nlargest(20, 'meijer_ton_yr')[['meijer_ton_yr', 'country_iso', 'plastic_ratio', 'E_plastic_corrected', 'rank_plastic']]
top20_corrected = df.nlargest(20, 'E_plastic_corrected')

print('TOP 20 RIVERS — after plastic_pct correction:')
print(f'{"New#":>5s} {"Old#":>5s} {"Dir":>3s} {"Meijer":>10s} {"×ratio":>6s} {"Corrected":>10s} {"Country":>7s}')
print('-'*55)
for i, (_, row) in enumerate(top20_corrected.iterrows()):
    old_rank = int(df.nlargest(int(row['rank_meijer']+1), 'meijer_ton_yr').index[0]) + 1 if row['rank_meijer'] <= 20 else int(row['rank_meijer'])
    # simpler: just use rank_meijer from original df
    meijer_rank = int(df.loc[df['E_plastic_corrected'] == row['E_plastic_corrected'], 'rank_meijer'].values[0]) if len(df.loc[df['E_plastic_corrected'] == row['E_plastic_corrected']]) > 0 else -1
    print(f'{i+1:5d} {int(row["rank_meijer"]):5d} {"↑" if i+1 < int(row["rank_meijer"]) else ("↓" if i+1 > int(row["rank_meijer"]) else "="):>3s} {row["meijer_ton_yr"]:>10,.0f} {row["plastic_ratio"]:>6.2f} {row["E_plastic_corrected"]:>10,.0f} {row["country_iso"]}')

In [ ]:
# Rank correlation
rho, p = stats.spearmanr(df['rank_meijer'], df['rank_plastic'])
print(f'Spearman ρ (Meijer vs plastic-corrected): {rho:.4f}')
print(f'Rank changes are modest globally (ρ≈0.99) but significant at the top.')

# But for TOP rivers, the changes matter
top100_meijer_set = set(df.nsmallest(100, 'rank_meijer').index)
top100_plastic_set = set(df.nsmallest(100, 'rank_plastic').index)
overlap = len(top100_meijer_set & top100_plastic_set)
print(f'\nTop 100 overlap: {overlap}/100 rivers are the same')
print(f'Top 100 changed: {100-overlap}/100 rivers differ')

## 3. Layer 2: Observational calibration

Apply the log-log calibration on top of the plastic-corrected emissions.
This addresses the structural overestimation (slope < 1 in log-log space).

In [ ]:
# First: re-fit the calibration using plastic-corrected emissions as the predictor
# instead of raw Meijer emissions
from scipy.spatial import cKDTree

obs_coords = np.column_stack([obs['lon'].values, obs['lat'].values])
df_coords = np.column_stack([df['lon'].values, df['lat'].values])
tree = cKDTree(df_coords)
dist, idx = tree.query(obs_coords)

# Get plastic-corrected emissions for observed rivers
obs_plastic_corrected = df.iloc[idx]['E_plastic_corrected'].values

# Fit calibration: log10(obs) = a + b * log10(E_plastic_corrected)
valid = obs_plastic_corrected > 0
log_obs = np.log10(obs['obs_annual'].values[valid])
log_pred = np.log10(obs_plastic_corrected[valid])

slope_pc, intercept_pc, r_value_pc, p_value_pc, std_err_pc = stats.linregress(log_pred, log_obs)
r2_pc = r_value_pc ** 2

print('Calibration on plastic-corrected emissions:')
print(f'  log₁₀(obs) = {slope_pc:.3f} × log₁₀(E_plastic) + {intercept_pc:.3f}')
print(f'  R² = {r2_pc:.3f}, p = {p_value_pc:.2e}')
print(f'  Slope 95% CI: [{slope_pc - 1.96*std_err_pc:.3f}, {slope_pc + 1.96*std_err_pc:.3f}]')
print()

# Compare with calibration on raw Meijer
log_meijer_obs = np.log10(obs['outfall_ME'].values[valid])
slope_m, intercept_m, r_m, _, se_m = stats.linregress(log_meijer_obs, log_obs)
print('Calibration on raw Meijer emissions (for reference):')
print(f'  log₁₀(obs) = {slope_m:.3f} × log₁₀(Meijer) + {intercept_m:.3f}')
print(f'  R² = {r_m**2:.3f}')
print()
print(f'Slope change: {slope_m:.3f} → {slope_pc:.3f} (plastic correction shifts slope)')

In [ ]:
# Apply the two-layer correction to all rivers
# Layer 1: plastic_pct correction (already done: E_plastic_corrected)
# Layer 2: observational calibration on plastic-corrected values

log_E_plastic = np.log10(df['E_plastic_corrected'].values.copy())
log_E_final = intercept_pc + slope_pc * log_E_plastic
df['E_final'] = 10 ** log_E_final
df['rank_final'] = df['E_final'].rank(ascending=False)

print('FINAL UPDATED RANKING (plastic correction + calibration):')
print(f'  Total: {df["E_final"].sum():,.0f} ton/yr')
print(f'  vs Meijer: {df["meijer_ton_yr"].sum():,.0f} ton/yr')
print(f'  vs Calibrated (no plastic): {df["calibrated_ton_yr"].sum():,.0f} ton/yr')
print()

# Concentration metrics
for label, col in [('Meijer', 'meijer_ton_yr'), ('Plastic-corrected', 'E_plastic_corrected'), ('Final (plastic+calib)', 'E_final')]:
    total = df[col].sum()
    top100 = df.nlargest(100, col)[col].sum()
    top1000 = df.nlargest(1000, col)[col].sum()
    cumsum = df.nlargest(len(df), col)[col].cumsum()
    n80 = (cumsum < 0.80 * total).sum() + 1
    print(f'{label:25s}: total={total:>10,.0f}  top100={top100/total*100:5.1f}%  top1000={top1000/total*100:5.1f}%  n80={n80:,}')

## 4. Top 20 comparison: Meijer vs Final

In [ ]:
top20_final = df.nlargest(20, 'E_final')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: Meijer top 20
top20_m = df.nlargest(20, 'meijer_ton_yr')
colors_m = top20_m['plastic_ratio'].values
axes[0].barh(range(20), top20_m['meijer_ton_yr'].values, color=plt.cm.RdYlGn_r(colors_m / colors_m.max()))
axes[0].set_yticks(range(20))
axes[0].set_yticklabels([f'{r["country_iso"]} #{i+1}' for i, (_, r) in enumerate(top20_m.iterrows())])
axes[0].invert_yaxis()
axes[0].set_xlabel('Emission (ton/yr)')
axes[0].set_title('Meijer et al. (2021) — Top 20')

# Right: Final top 20
colors_f = top20_final['plastic_ratio'].values
axes[1].barh(range(20), top20_final['E_final'].values, color=plt.cm.RdYlGn_r(colors_f / colors_f.max()))
axes[1].set_yticks(range(20))
axes[1].set_yticklabels([f'{r["country_iso"]} #{i+1}' for i, (_, r) in enumerate(top20_final.iterrows())])
axes[1].invert_yaxis()
axes[1].set_xlabel('Emission (ton/yr)')
axes[1].set_title('Updated (WaW 3.0 + Calibration) — Top 20')

plt.tight_layout()
plt.savefig(RESULTS / 'fig6_top20_comparison.png')
plt.show()
print('Color: green = plastic_ratio < 1 (Meijer overestimates), red = > 1 (Meijer underestimates)')

## 5. Capture rate implications

In [ ]:
# For TOC: what fraction of emissions is captured by interceptors at top rivers?
# Under Meijer, if an interceptor captures X ton/yr at a river rated at E_meijer,
# the capture rate is X / E_meijer.
# Under our update, the capture rate is X / E_final.
# Since E_final < E_meijer for most top rivers, capture rates are HIGHER.

# For top 1000 emitters
top1k = df.nlargest(1000, 'meijer_ton_yr')
capture_boost = top1k['meijer_ton_yr'] / top1k['E_final']

print('CAPTURE RATE BOOST (top 1000 Meijer emitters):')
print(f'  Mean: {capture_boost.mean():.2f}x')
print(f'  Median: {capture_boost.median():.2f}x')
print(f'  Range: [{capture_boost.min():.2f}x, {capture_boost.max():.2f}x]')
print(f'  P(capture boost > 1.0): {(capture_boost > 1.0).mean()*100:.1f}%')
print()
print('Interpretation: interceptors at top rivers capture {:.1f}x more than'.format(capture_boost.median()))
print('reported, because Meijer overestimates the emission at those rivers.')

## 6. Sensitivity: What if we're wrong about plastic_pct?

In [ ]:
# Test scenarios: plastic_pct correction at 50% and 150% of WaW 3.0 values
scenarios = {
    'No plastic correction': 1.0,
    'WaW 3.0 × 0.5': 0.5,
    'WaW 3.0 (full)': 1.0,
    'WaW 3.0 × 1.5': 1.5,
}

print('SENSITIVITY ANALYSIS — plastic_pct correction magnitude:')
print(f'{"Scenario":25s} {"Total":>10s} {"Top1000":>7s} {"n80":>7s}')
print('-'*55)

for label, multiplier in scenarios.items():
    if multiplier == 1.0 and label == 'No plastic correction':
        E_scenario = df['meijer_ton_yr'].values
    else:
        # plastic_ratio relative to 12%, scaled by multiplier
        adjusted_ratio = 1.0 + (df['plastic_ratio'].values - 1.0) * multiplier
        E_scenario = df['meijer_ton_yr'].values * adjusted_ratio
    
    # Apply calibration
    log_E = np.log10(np.clip(E_scenario, 1e-10, None))
    E_cal = 10 ** (intercept_pc + slope_pc * log_E)
    
    total = E_cal.sum()
    top1k = np.sort(E_cal)[::-1][:1000].sum()
    cs = np.sort(E_cal)[::-1].cumsum()
    n80 = (cs < 0.80 * total).sum() + 1
    
    print(f'{label:25s} {total:>10,.0f} {top1k/total*100:>6.1f}% {n80:>7,}')

## 7. Bootstrap uncertainty on final ranking

In [ ]:
np.random.seed(42)
n_boot = 2000

# Get observed data matched to plastic-corrected values
log_obs_vals = np.log10(obs['obs_annual'].values)
log_pred_vals = np.log10(obs_plastic_corrected)
valid_mask = np.isfinite(log_obs_vals) & np.isfinite(log_pred_vals)
log_obs_clean = log_obs_vals[valid_mask]
log_pred_clean = log_pred_vals[valid_mask]

top1000_pcts = []
n80s = []

for i in range(n_boot):
    idx_boot = np.random.choice(len(log_obs_clean), len(log_obs_clean), replace=True)
    s, ic, _, _, _ = stats.linregress(log_pred_clean[idx_boot], log_obs_clean[idx_boot])
    
    log_cal = ic + s * np.log10(df['E_plastic_corrected'].values.copy())
    cal = 10 ** log_cal
    
    total = cal.sum()
    top1k = np.sort(cal)[::-1][:1000].sum()
    top1000_pcts.append(top1k / total * 100)
    
    cs = np.sort(cal)[::-1].cumsum()
    n80 = (cs < 0.80 * total).sum() + 1
    n80s.append(n80)

top1000_pcts = np.array(top1000_pcts)
n80s = np.array(n80s)

print('BOOTSTRAP UNCERTAINTY (n=2000):')
print(f'  Top 1000 concentration: {np.median(top1000_pcts):.1f}% [{np.percentile(top1000_pcts,2.5):.1f}%, {np.percentile(top1000_pcts,97.5):.1f}%]')
print(f'  Rivers for 80%: {np.median(n80):,.0f} [{np.percentile(n80s,2.5):,.0f}, {np.percentile(n80s,97.5):,.0f}]')
print()
print('DIRECTION ROBUSTNESS:')
print(f'  P(top 1000 < 71.8%): {(top1000_pcts < 71.8).mean()*100:.1f}%')
print(f'  P(rivers for 80% > 1,659): {(n80s > 1659).mean()*100:.1f}%')

## 8. Japan sensitivity

In [ ]:
# Re-fit calibration without Japan
is_japan_obs = obs['country'].values == 'JPN'
non_japan_mask = valid_mask & ~is_japan_obs
log_obs_nj = log_obs_vals[non_japan_mask]
log_pred_nj = log_pred_vals[non_japan_mask]

slope_nj, intercept_nj, r_nj, _, se_nj = stats.linregress(log_pred_nj, log_obs_nj)

print('CALIBRATION WITHOUT JAPAN:')
print(f'  log₁₀(obs) = {slope_nj:.3f} × log₁₀(E_plastic) + {intercept_nj:.3f}')
print(f'  R² = {r_nj**2:.3f}, n={non_japan_mask.sum()}')
print()

# Apply no-Japan calibration
log_E_final_nj = intercept_nj + slope_nj * np.log10(df['E_plastic_corrected'].values.copy())
E_final_nj = 10 ** log_E_final_nj

total_nj = E_final_nj.sum()
top1k_nj = np.sort(E_final_nj)[::-1][:1000].sum()
cs_nj = np.sort(E_final_nj)[::-1].cumsum()
n80_nj = (cs_nj < 0.80 * total_nj).sum() + 1

print('RESULTS WITHOUT JAPAN CALIBRATION:')
print(f'  Total: {total_nj:,.0f} ton/yr')
print(f'  Top 1000: {top1k_nj/total_nj*100:.1f}%')
print(f'  Rivers for 80%: {n80_nj:,}')
print()
print('Even without Japan, the direction holds:')
print(f'  Top 1000 = {top1k_nj/total_nj*100:.1f}% (not 71.8%)')
print(f'  Rivers for 80% = {n80_nj:,} (not 1,659)')

## 9. Export final ranking

In [ ]:
export = df[['lon', 'lat', 'meijer_ton_yr', 'E_plastic_corrected', 'E_final',
             'country_iso', 'plastic_ratio', 'plastic_pct',
             'rank_meijer', 'rank_plastic', 'rank_final',
             'DIS_AV_CMS', 'ORD_STRA', 'LENGTH_KM', 'UPLAND_SKM']].copy()

export.columns = ['lon', 'lat', 'meijer_ton_yr', 'plastic_corrected_ton_yr', 'final_ton_yr',
                  'country_iso', 'plastic_ratio', 'plastic_pct',
                  'rank_meijer', 'rank_plastic_corrected', 'rank_final',
                  'discharge_cms', 'strahler_order', 'length_km', 'upstream_area_km2']

export.to_csv(DATA_PROC / 'final_ranking_updated.csv', index=False)
print(f'Exported {len(export):,} rivers to final_ranking_updated.csv')
print()
print('SUMMARY:')
print(f'  Meijer total:       {df["meijer_ton_yr"].sum():>12,.0f} ton/yr')
print(f'  Plastic-corrected:  {df["E_plastic_corrected"].sum():>12,.0f} ton/yr')
print(f'  Final (plastic+cal):{df["E_final"].sum():>12,.0f} ton/yr')